# ENVIRONMENT SETUP & LIBRARIES IMPORT

In [1]:
!pip install transformers datasets accelerate evaluate torch scikit-learn pandas numpy matplotlib seaborn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00


In [2]:
# ── Standard libraries ────────────────────────────────────────────
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report,confusion_matrix,f1_score,accuracy_score)

# ── PyTorch ───────────────────────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# ── HuggingFace ───────────────────────────────────────────────────
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import evaluate

# ── Reproducibility ───────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Device ────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


# Load FinBERT from HuggingFace

In [3]:
MODEL_NAME = "ProsusAI/finbert"

# ── Load tokenizer ────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Vocab size      : {tokenizer.vocab_size}")
print(f"Max length      : {tokenizer.model_max_length}")

# ── Load model ────────────────────────────────────────────────────
# num_labels=3 because we have positive / negative / neutral
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    ignore_mismatched_sizes=True   # handles classifier head replacement
)
model = model.to(device)
print(f"\nModel loaded: {MODEL_NAME}")
print(f"Parameters  : {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable   : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded: ProsusAI/finbert
Vocab size      : 30522
Max length      : 512


pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Model loaded: ProsusAI/finbert
Parameters  : 109,484,547
Trainable   : 109,484,547


In [4]:
LABEL2ID = {
    "negative": 0,
    "neutral" : 1,
    "positive": 2
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# Update the model config to reflect your mapping
model.config.label2id = LABEL2ID
model.config.id2label = ID2LABEL

print("Label mapping locked:")
for label, idx in LABEL2ID.items():
    print(f"  {idx} → {label}")

Label mapping locked:
  0 → negative
  1 → neutral
  2 → positive


In [5]:
test_headline = "Reliance Industries reports record quarterly profit"

inputs = tokenizer(
    test_headline,
    return_tensors="pt",
    truncation=True,
    max_length=128,
    padding=True
).to(device)

with torch.no_grad():
    outputs = model(**inputs)
    probs   = torch.softmax(outputs.logits, dim=1).squeeze()
    pred_id = torch.argmax(probs).item()

print(f"Headline  : {test_headline}")
print(f"Prediction: {ID2LABEL[pred_id]}")
print(f"\nClass probabilities:")
for label, prob in zip(LABEL2ID.keys(), probs.tolist()):
    bar = '█' * int(prob * 30)
    print(f"  {label:<10} {prob:.4f}  {bar}")

Headline  : Reliance Industries reports record quarterly profit
Prediction: negative

Class probabilities:
  negative   0.7534  ██████████████████████
  neutral    0.1807  █████
  positive   0.0659  █


## DATASET LOAD

In [6]:
# ── Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Base paths ────────────────────────────────────────────────────
BASE  = "/content/drive/MyDrive/finbert_project/data/"
MDIR  = "/content/drive/MyDrive/finbert_project/models/"

import os
os.makedirs(MDIR, exist_ok=True)

# ── Load all datasets ─────────────────────────────────────────────
import pandas as pd

m1_train = pd.read_csv(f"{BASE}m1_sentfin_train.csv")
m1_val   = pd.read_csv(f"{BASE}m1_sentfin_val.csv")

m2_train = pd.read_csv(f"{BASE}m2_indian_train.csv")
m2_val   = pd.read_csv(f"{BASE}m2_indian_val.csv")

m3_train = pd.read_csv(f"{BASE}m3_combined_train.csv")
m3_val   = pd.read_csv(f"{BASE}m3_combined_val.csv")

test_sf  = pd.read_csv(f"{BASE}test_sentfin.csv")
test_in  = pd.read_csv(f"{BASE}test_indian.csv")

# ── Verify everything loaded correctly ────────────────────────────
datasets = {
    "M1 Train"   : m1_train,
    "M1 Val"     : m1_val,
    "M2 Train"   : m2_train,
    "M2 Val"     : m2_val,
    "M3 Train"   : m3_train,
    "M3 Val"     : m3_val,
    "Test SEntFiN": test_sf,
    "Test Indian" : test_in,
}
print("  DATASETS LOADED")
for name, df in datasets.items():
    dist = df['final_label'].value_counts().to_dict()
    print(f"  {name:<15} : {len(df):>5} rows  {dist}")

Mounted at /content/drive
  DATASETS LOADED
  M1 Train        :  5861 rows  {'positive': 2163, 'neutral': 1984, 'negative': 1714}
  M1 Val          :   733 rows  {'positive': 271, 'neutral': 248, 'negative': 214}
  M2 Train        :  3075 rows  {'positive': 1305, 'neutral': 1045, 'negative': 725}
  M2 Val          :   384 rows  {'positive': 163, 'neutral': 130, 'negative': 91}
  M3 Train        :  8936 rows  {'positive': 3468, 'neutral': 3029, 'negative': 2439}
  M3 Val          :  1117 rows  {'positive': 434, 'neutral': 378, 'negative': 305}
  Test SEntFiN    :   733 rows  {'positive': 270, 'neutral': 248, 'negative': 215}
  Test Indian     :   385 rows  {'positive': 164, 'neutral': 131, 'negative': 90}


In [8]:
from torch.utils.data import Dataset
import torch

class FinBERTDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        # Tokenize all headlines at once (faster than one at a time)
        self.encodings = tokenizer(
            df['headline'].tolist(),
            max_length      = max_length,
            padding         = 'max_length',
            truncation      = True,       # safety net — won't fire but needed
            return_tensors  = 'pt'
        )
        # Convert text labels → numbers: negative=0, neutral=1, positive=2
        self.labels = torch.tensor(
            df['final_label'].map(LABEL2ID).tolist(),
            dtype = torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Return one sample — model needs exactly these 3 things
        return {
            'input_ids'     : self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels'        : self.labels[idx]
        }


# ── Create all 8 dataset objects ──────────────────────────────────
MAX_LEN = 128

ds_m1_train = FinBERTDataset(m1_train, tokenizer, MAX_LEN)
ds_m1_val   = FinBERTDataset(m1_val,   tokenizer, MAX_LEN)
ds_m2_train = FinBERTDataset(m2_train, tokenizer, MAX_LEN)
ds_m2_val   = FinBERTDataset(m2_val,   tokenizer, MAX_LEN)
ds_m3_train = FinBERTDataset(m3_train, tokenizer, MAX_LEN)
ds_m3_val   = FinBERTDataset(m3_val,   tokenizer, MAX_LEN)
ds_test_sf  = FinBERTDataset(test_sf,  tokenizer, MAX_LEN)
ds_test_in  = FinBERTDataset(test_in,  tokenizer, MAX_LEN)

# MODEL TRAINING

In [9]:
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import evaluate
import numpy as np

# ══════════════════════════════════════════════════════════════════
# STEP 1 — METRICS FUNCTION
# Called after every epoch to compute F1, accuracy on val set
# ══════════════════════════════════════════════════════════════════
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # logits = raw model outputs (not probabilities yet)
    # take the highest value across 3 classes as prediction
    predictions = np.argmax(logits, axis=1)

    # weighted F1 — accounts for class imbalance
    f1 = f1_score(labels, predictions,average='weighted',labels=[0,1,2])

    acc = accuracy_score(labels, predictions)

    return {
        "f1"      : round(f1,  4),
        "accuracy": round(acc, 4)
    }


# ══════════════════════════════════════════════════════════════════
# STEP 2 — TRAINING ARGUMENTS
# All hyperparameters in one place
# ══════════════════════════════════════════════════════════════════
def get_training_args(output_dir, epochs=10 , learning_rate = 2e-5):
    return TrainingArguments(

        output_dir    = output_dir,   # where to save checkpoints

        # ── Training ──────────────────────────────────────────────
        num_train_epochs              = epochs,
        per_device_train_batch_size   = 16,   # 16 headlines per step
        per_device_eval_batch_size    = 32,   # larger ok for eval (no gradients)
        learning_rate                 = learning_rate,
        weight_decay                  = 0.01, # mild regularisation

        # ── Warmup ────────────────────────────────────────────────
        # Slowly ramp up learning rate for first 10% of steps
        # prevents large updates from destroying pretrained weights
        warmup_steps                = 200,

        # ── Evaluation ────────────────────────────────────────────
        eval_strategy                 = "epoch",  # eval after every epoch
        save_strategy                 = "epoch",  # save after every epoch
        load_best_model_at_end        = True,     # auto-load best checkpoint
        metric_for_best_model         = "f1",     # use F1 to pick best
        greater_is_better             = True,

        # ── Logging ───────────────────────────────────────────────
        logging_steps                 = 50,       # print loss every 50 steps
        report_to                     = "none",   # disable wandb/tensorboard

        # ── Reproducibility ───────────────────────────────────────
        seed                          = 42,
        fp16                          = True,     # faster training on Colab GPU
    )


# ══════════════════════════════════════════════════════════════════
# STEP 3 — TRAIN ONE MODEL
# Reusable function — call it 3 times for M1, M2, M3
# ══════════════════════════════════════════════════════════════════
def train_model(model_name, train_dataset, val_dataset,
                output_dir, epochs=10, learning_rate=2e-5):

    print(f"\n{'█'*55}")
    print(f"  Training: {model_name}")
    print(f"  Train samples : {len(train_dataset)}")
    print(f"  Val samples   : {len(val_dataset)}")
    print(f"  Learning rate : {learning_rate}")
    print(f"  Epochs (max)  : {epochs}")
    print(f"{'█'*55}\n")

    # Load a fresh FinBERT for each model
    # IMPORTANT: fresh load every time so models don't share weights
    model = AutoModelForSequenceClassification.from_pretrained(
        "ProsusAI/finbert",
        num_labels             = 3,
        id2label               = ID2LABEL,
        label2id               = LABEL2ID,
        ignore_mismatched_sizes= True
    )

    # Trainer — handles training loop, eval, checkpointing
    trainer = Trainer(
        model           = model,
        args            = get_training_args(output_dir, epochs, learning_rate),
        train_dataset   = train_dataset,
        eval_dataset    = val_dataset,
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)]
    )

    # Train
    trainer.train()

    # Save final best model to Google Drive
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"\n  ✓ Model saved → {output_dir}")

    return trainer

In [44]:
# ══════════════════════════════════════════════════════════════════
# STEP 4 — RUN ALL 3 MODELS
# Train one at a time — each takes ~15–25 min on Colab T4
# ══════════════════════════════════════════════════════════════════

# Model 1 — SEntFiN only
trainer_m1 = train_model(
    model_name    = "Model 1 — SEntFiN only",
    train_dataset = ds_m1_train,
    val_dataset   = ds_m1_val,
    output_dir    = f"{MDIR}model1_sentfin",
    learning_rate = 1e-5,
    epochs        = 10
)



███████████████████████████████████████████████████████
  Training: Model 1 — SEntFiN only
  Train samples : 5861
  Val samples   : 733
  Learning rate : 1e-05
  Epochs (max)  : 10
███████████████████████████████████████████████████████



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.541676,0.560025,0.775200,0.773500
2,0.435713,0.446068,0.831100,0.830800
3,0.248083,0.452913,0.849200,0.848600
4,0.169818,0.545695,0.848700,0.848600
5,0.168615,0.626966,0.845500,0.845800
6,0.082433,0.683667,0.862100,0.862200
7,0.081451,0.756791,0.850200,0.849900
8,0.029776,0.780930,0.852800,0.852700
9,0.025344,0.829080,0.851200,0.851300


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  ✓ Model saved → /content/drive/MyDrive/finbert_project/models/model1_sentfin


In [45]:
# Model 2 — Indian only
trainer_m2 = train_model(
    model_name    = "Model 2 — Indian only",
    train_dataset = ds_m2_train,
    val_dataset   = ds_m2_val,
    output_dir    = f"{MDIR}model2_indian",
    learning_rate = 5e-6,    # lowest — smallest dataset
    epochs        = 10
)


███████████████████████████████████████████████████████
  Training: Model 2 — Indian only
  Train samples : 3075
  Val samples   : 384
  Learning rate : 5e-06
  Epochs (max)  : 10
███████████████████████████████████████████████████████



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,1.098048,0.803859,0.602800,0.627600
2,0.599500,0.580578,0.758700,0.760400
3,0.375976,0.516769,0.792700,0.794300
4,0.332345,0.502392,0.822300,0.822900
5,0.237519,0.486507,0.824800,0.825500
6,0.205634,0.520316,0.816700,0.817700
7,0.152619,0.538808,0.814700,0.815100
8,0.112595,0.562643,0.824900,0.825500
9,0.119934,0.560932,0.827700,0.828100
10,0.093394,0.562553,0.825200,0.825500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  ✓ Model saved → /content/drive/MyDrive/finbert_project/models/model2_indian


In [46]:
# Model 3 — Combined
trainer_m3 = train_model(
    model_name    = "Model 3 — Combined",
    train_dataset = ds_m3_train,
    val_dataset   = ds_m3_val,
    output_dir    = f"{MDIR}model3_combined",
    learning_rate = 2e-5,
    epochs        = 10
)


███████████████████████████████████████████████████████
  Training: Model 3 — Combined
  Train samples : 8936
  Val samples   : 1117
  Learning rate : 2e-05
  Epochs (max)  : 10
███████████████████████████████████████████████████████



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.439119,0.398856,0.834400,0.834400
2,0.293365,0.409975,0.844700,0.846000
3,0.203243,0.523101,0.849800,0.849600
4,0.128326,0.662483,0.856100,0.856800
5,0.058461,0.829174,0.852700,0.852300
6,0.029864,0.897423,0.857300,0.856800
7,0.035847,0.940950,0.853200,0.853200
8,0.009417,0.960314,0.859900,0.859400
9,0.016866,1.000206,0.856000,0.855900
10,0.000244,1.006026,0.858200,0.857700


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  ✓ Model saved → /content/drive/MyDrive/finbert_project/models/model3_combined


# OPTIMIZING OVERFITTING :

In [10]:
# Load tokenizer from your saved model — no internet needed
tokenizer = AutoTokenizer.from_pretrained(f"{MDIR}model1_sentfin")
print("Tokenizer loaded")

Tokenizer loaded


In [16]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score
)
import matplotlib.pyplot as plt
import seaborn as sns

# ══════════════════════════════════════════════════════════════════
# STEP 1 — EVALUATE ONE MODEL ON ONE TEST SET
# Returns predictions, true labels, and all metrics
# ══════════════════════════════════════════════════════════════════
def evaluate_model(model_path, test_dataset, test_df, model_label):

    print(f"\n  Loading {model_label} from {model_path}...")

    # Load saved model and tokenizer
    model     = AutoModelForSequenceClassification.from_pretrained(model_path)
    model     = model.to(device)
    model.eval()   # evaluation mode — disables dropout

    # DataLoader — batches test data for efficient inference
    loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    all_preds  = []
    all_labels = []

    # No gradient computation needed during evaluation — saves memory
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            # Forward pass — get raw logits
            outputs = model(input_ids=input_ids,
                           attention_mask=attention_mask)

            # Take highest logit as prediction
            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Convert to arrays
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Compute metrics
    f1  = f1_score(all_labels, all_preds, average='weighted')
    acc = accuracy_score(all_labels, all_preds)

    print(f"\n  {'─'*50}")
    print(f"  Model  : {model_label}")
    print(f"  {'─'*50}")
    print(f"  Weighted F1 : {f1:.4f}")
    print(f"  Accuracy    : {acc:.4f}")
    print(f"\n  Per-class report:")
    print(classification_report(
        all_labels, all_preds,
        target_names = [ID2LABEL[i] for i in range(3)],
        digits       = 4
    ))

    return {
        "model"      : model_label,
        "f1"         : round(f1,  4),
        "accuracy"   : round(acc, 4),
        "preds"      : all_preds,
        "labels"     : all_labels,
    }


# ══════════════════════════════════════════════════════════════════
# STEP 2 — EVALUATE VANILLA FINBERT (no fine-tuning)
# Uses original FinBERT label mapping — must remap to yours
# ══════════════════════════════════════════════════════════════════
def evaluate_vanilla(test_dataset, model_label):

    print(f"\n  Loading Vanilla FinBERT...")

    # Load original FinBERT — NOT from your saved models
    model = AutoModelForSequenceClassification.from_pretrained(
        "ProsusAI/finbert"
    ).to(device)
    model.eval()

    # Vanilla FinBERT's original label mapping
    # DIFFERENT from your mapping — must remap outputs
    VANILLA_ID2LABEL = {0: "positive", 1: "negative", 2: "neutral"}

    loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids,
                           attention_mask=attention_mask)
            preds   = torch.argmax(outputs.logits, dim=1)

            # Remap vanilla predictions to YOUR label scheme
            # vanilla: 0=pos,1=neg,2=neu → yours: 0=neg,1=neu,2=pos
            remapped = []
            for p in preds.cpu().numpy():
                original_label = VANILLA_ID2LABEL[p]   # get label name
                remapped.append(LABEL2ID[original_label])  # map to your id

            all_preds.extend(remapped)
            all_labels.extend(labels.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    f1  = f1_score(all_labels, all_preds, average='weighted')
    acc = accuracy_score(all_labels, all_preds)

    print(f"\n  {'─'*50}")
    print(f"  Model  : {model_label}")
    print(f"  {'─'*50}")
    print(f"  Weighted F1 : {f1:.4f}")
    print(f"  Accuracy    : {acc:.4f}")
    print(f"\n  Per-class report:")
    print(classification_report(
        all_labels, all_preds,
        target_names = [ID2LABEL[i] for i in range(3)],
        digits       = 4
    ))

    return {
        "model"   : model_label,
        "f1"      : round(f1,  4),
        "accuracy": round(acc, 4),
        "preds"   : all_preds,
        "labels"  : all_labels,
    }


# ══════════════════════════════════════════════════════════════════
# STEP 3 — RUN ALL EVALUATIONS
# 4 models × 2 test sets = 8 results
# ══════════════════════════════════════════════════════════════════
print("█"*55)
print("  EVALUATION — ALL MODELS ON BOTH TEST SETS")
print("█"*55)

# ── Test set 1: SEntFiN ───────────────────────────────────────────
print("\n" + "═"*55)
print("  TEST SET 1 — SEntFiN (733 rows)")
print("═"*55)

sf_vanilla = evaluate_vanilla(ds_test_sf, "Vanilla FinBERT")
sf_m1      = evaluate_model(f"{MDIR}model1_sentfin", ds_test_sf, test_sf, "Model 1 — SEntFiN")
sf_m2      = evaluate_model(f"{MDIR}model2_indian",  ds_test_sf, test_sf, "Model 2 — Indian")
sf_m3      = evaluate_model(f"{MDIR}model3_combined",ds_test_sf, test_sf, "Model 3 — Combined")

# ── Test set 2: Indian ────────────────────────────────────────────
print("\n" + "═"*55)
print("  TEST SET 2 — Indian (385 rows)")
print("═"*55)

in_vanilla = evaluate_vanilla(ds_test_in, "Vanilla FinBERT")
in_m1      = evaluate_model(f"{MDIR}model1_sentfin", ds_test_in, test_in, "Model 1 — SEntFiN")
in_m2      = evaluate_model(f"{MDIR}model2_indian",  ds_test_in, test_in, "Model 2 — Indian")
in_m3      = evaluate_model(f"{MDIR}model3_combined",ds_test_in, test_in, "Model 3 — Combined")


# ══════════════════════════════════════════════════════════════════
# STEP 4 — RESULTS TABLE
# Clean summary of all 8 scores
# ══════════════════════════════════════════════════════════════════
results = pd.DataFrame([
    {"Model": "Vanilla FinBERT",    "SEntFiN F1": sf_vanilla["f1"], "Indian F1": in_vanilla["f1"],
                                     "SEntFiN Acc": sf_vanilla["accuracy"], "Indian Acc": in_vanilla["accuracy"]},
    {"Model": "M1 — SEntFiN only",  "SEntFiN F1": sf_m1["f1"],     "Indian F1": in_m1["f1"],
                                     "SEntFiN Acc": sf_m1["accuracy"],     "Indian Acc": in_m1["accuracy"]},
    {"Model": "M2 — Indian only",   "SEntFiN F1": sf_m2["f1"],     "Indian F1": in_m2["f1"],
                                     "SEntFiN Acc": sf_m2["accuracy"],     "Indian Acc": in_m2["accuracy"]},
    {"Model": "M3 — Combined",      "SEntFiN F1": sf_m3["f1"],     "Indian F1": in_m3["f1"],
                                     "SEntFiN Acc": sf_m3["accuracy"],     "Indian Acc": in_m3["accuracy"]},
])

print("\n" + "═"*65)
print("  FINAL RESULTS TABLE")
print("═"*65)
print(results.to_string(index=False))
print("═"*65)

# Save results
results.to_csv(f"{MDIR}evaluation_results.csv", index=False)
print(f"\n  Saved → {MDIR}evaluation_results.csv")


# ══════════════════════════════════════════════════════════════════
# STEP 5 — CONFUSION MATRICES (one per model per test set)
# ══════════════════════════════════════════════════════════════════
def plot_confusion_matrix(result, title):
    cm  = confusion_matrix(result["labels"], result["preds"])
    pct = cm / cm.sum(axis=1, keepdims=True) * 100

    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        pct, annot=False, fmt='.1f', cmap='Blues',
        xticklabels=[ID2LABEL[i] for i in range(3)],
        yticklabels=[ID2LABEL[i] for i in range(3)],
        ax=ax
    )
    # Annotate cells with count + percentage
    for i in range(3):
        for j in range(3):
            ax.text(j+0.5, i+0.5,
                    f"{cm[i,j]}\n({pct[i,j]:.1f}%)",
                    ha='center', va='center',
                    fontsize=9,
                    color='white' if pct[i,j] > 50 else 'black')

    ax.set_title(f"{title}\nF1={result['f1']:.4f}  Acc={result['accuracy']:.4f}",
                 fontsize=11, pad=10)
    ax.set_xlabel("Predicted", fontsize=10)
    ax.set_ylabel("Actual",    fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{MDIR}{title.replace(' ','_').replace('—','')}.png",
                dpi=150, bbox_inches='tight')
    plt.show()

# Plot all 8 confusion matrices
for result, title in [
    (sf_vanilla, "Vanilla — SEntFiN Test"),
    (sf_m1,      "M1 — SEntFiN Test"),
    (sf_m2,      "M2 — SEntFiN Test"),
    (sf_m3,      "M3 — SEntFiN Test"),
    (in_vanilla, "Vanilla — Indian Test"),
    (in_m1,      "M1 — Indian Test"),
    (in_m2,      "M2 — Indian Test"),
    (in_m3,      "M3 — Indian Test"),
]:
    plot_confusion_matrix(result, title)

███████████████████████████████████████████████████████
  EVALUATION — ALL MODELS ON BOTH TEST SETS
███████████████████████████████████████████████████████

═══════════════════════════════════════════════════════
  TEST SET 1 — SEntFiN (733 rows)
═══════════════════════════════════════════════════════

  Loading Vanilla FinBERT...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [12]:
from statsmodels.stats.contingency_tables import mcnemar as mc_test
import numpy as np

def mcnemar_test(preds_a, preds_b, labels, name_a, name_b):
    correct_a = (np.array(preds_a) == np.array(labels))
    correct_b = (np.array(preds_b) == np.array(labels))

    # b = A correct, B wrong
    # c = A wrong,   B correct
    b = np.sum( correct_a & ~correct_b)
    c = np.sum(~correct_a &  correct_b)

    table  = [[0, b], [c, 0]]
    result = mc_test(table, exact=False, correction=True)

    sig = "✓ significant" if result.pvalue < 0.05 else "✗ not significant"
    print(f"\n  {name_a} vs {name_b}")
    print(f"  b={b}  c={c}  p={result.pvalue:.4f}  {sig}")


print("="*55)
print("  MCNEMAR SIGNIFICANCE TESTS")
print("="*55)

# M3 vs M1 on SEntFiN — is combined better than SEntFiN-only?
mcnemar_test(sf_m3['preds'], sf_m1['preds'],
             sf_m3['labels'], "M3", "M1 — SEntFiN test")

# M3 vs M2 on Indian — is combined better than Indian-only?
mcnemar_test(in_m3['preds'], in_m2['preds'],
             in_m3['labels'], "M3", "M2 — Indian test")

# M3 vs Vanilla on SEntFiN
mcnemar_test(sf_m3['preds'], sf_vanilla['preds'],
             sf_m3['labels'], "M3", "Vanilla — SEntFiN test")

# M3 vs Vanilla on Indian
mcnemar_test(in_m3['preds'], in_vanilla['preds'],
             in_m3['labels'], "M3", "Vanilla — Indian test")

print("="*55)

  MCNEMAR SIGNIFICANCE TESTS

  M3 vs M1 — SEntFiN test
  b=31  c=22  p=0.2718  ✗ not significant

  M3 vs M2 — Indian test
  b=19  c=18  p=1.0000  ✗ not significant

  M3 vs Vanilla — SEntFiN test
  b=132  c=35  p=0.0000  ✓ significant

  M3 vs Vanilla — Indian test
  b=59  c=25  p=0.0003  ✓ significant


In [11]:
from transformers import AutoConfig
# ── Metrics function ──────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=1)
    f1             = f1_score(labels, predictions, average='weighted')
    acc            = accuracy_score(labels, predictions)
    return {
        "f1"      : round(f1,  4),
        "accuracy": round(acc, 4)
    }

In [10]:
# CODE TO RUN AS GPU RESETS
def train_model_v2(model_name, train_dataset, val_dataset,
                   output_dir, epochs=10,
                   learning_rate=2e-5,
                   dropout=0.2,
                   freeze_layers=4,
                   weight_decay=0.1):

    print(f"\n{'█'*55}")
    print(f"  Training  : {model_name}")
    print(f"  LR        : {learning_rate}")
    print(f"  Dropout   : {dropout}")
    print(f"  Frozen    : bottom {freeze_layers} layers")
    print(f"  WD        : {weight_decay}")
    print(f"{'█'*55}\n")

    # Load model with increased dropout
    config = AutoConfig.from_pretrained(
        "ProsusAI/finbert",
        num_labels                   = 3,
        id2label                     = ID2LABEL,
        label2id                     = LABEL2ID,
        hidden_dropout_prob          = dropout,
        attention_probs_dropout_prob = dropout,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        "ProsusAI/finbert",
        config                  = config,
        ignore_mismatched_sizes = True
    )

    # Freeze bottom layers
    for param in model.bert.embeddings.parameters():
        param.requires_grad = False
    for i in range(freeze_layers):
        for param in model.bert.encoder.layer[i].parameters():
            param.requires_grad = False

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable params : {trainable:,}")

    args = TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = epochs,
        per_device_train_batch_size = 32,      # increased from 16
        per_device_eval_batch_size  = 64,
        learning_rate               = learning_rate,
        weight_decay                = weight_decay,
        warmup_steps                = 200,
        lr_scheduler_type           = "cosine", # smooth lr decay
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "f1",
        greater_is_better           = True,
        logging_steps               = 50,
        report_to                   = "none",
        seed                        = 42,
        fp16                        = True,
    )

    trainer = Trainer(
        model           = model,
        args            = args,
        train_dataset   = train_dataset,
        eval_dataset    = val_dataset,
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)]
    )

    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"\n  ✓ Saved → {output_dir}")

    return trainer


# ── Retrain all 3 models with anti-overfit settings ───────────────

# ── Priority 1 — M3 V2 (most important) ──────────────────────────
trainer_m3 = train_model_v2(
    model_name    = "Model 3 — Combined v2",
    train_dataset = ds_m3_train,
    val_dataset   = ds_m3_val,
    output_dir    = f"{MDIR}model3_combined_v2",
    learning_rate = 2e-5,
    dropout       = 0.2,
    freeze_layers = 4,
    weight_decay  = 0.1,
    epochs        = 10
)

# ── Priority 2 — M1 V3 ───────────────────────────────────────────
trainer_m1 = train_model_v2(
    model_name    = "Model 1 — SEntFiN v3",
    train_dataset = ds_m1_train,
    val_dataset   = ds_m1_val,
    output_dir    = f"{MDIR}model1_sentfin_v3",
    learning_rate = 1e-5,
    dropout       = 0.2,
    freeze_layers = 3,
    weight_decay  = 0.05,
    epochs        = 10
)

# ── Priority 3 — M2 V3 ───────────────────────────────────────────
trainer_m2 = train_model_v2(
    model_name    = "Model 2 — Indian v3",
    train_dataset = ds_m2_train,
    val_dataset   = ds_m2_val,
    output_dir    = f"{MDIR}model2_indian_v3",
    learning_rate = 1e-5,
    dropout       = 0.2,
    freeze_layers = 4,
    weight_decay  = 0.05,
    epochs        = 10
)


███████████████████████████████████████████████████████
  Training  : Model 3 — Combined v2
  LR        : 2e-05
  Dropout   : 0.2
  Frozen    : bottom 4 layers
  WD        : 0.1
███████████████████████████████████████████████████████



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Trainable params : 57,295,875


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.569893,0.512620,0.802200,0.802100
2,0.408834,0.409120,0.838500,0.839700
3,0.362826,0.404005,0.859300,0.859400
4,0.311275,0.411408,0.849400,0.850500
5,0.268070,0.395622,0.870600,0.870200
6,0.236874,0.408265,0.866500,0.865700
7,0.210125,0.425728,0.878200,0.878200
8,0.162127,0.434492,0.878500,0.878200
9,0.166922,0.434126,0.877500,0.877400
10,0.162131,0.434722,0.876700,0.876500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  ✓ Saved → /content/drive/MyDrive/finbert_project/models/model2_indian_v3


In [11]:
import os
MDIR = "/content/drive/MyDrive/finbert_project/models/"

for folder in sorted(os.listdir(MDIR)):
    folder_path = os.path.join(MDIR, folder)
    if os.path.isdir(folder_path):
        files = os.listdir(folder_path)
        has_weights = any(f.endswith('.safetensors') or
                         f.endswith('.bin') for f in files)
        has_config  = 'config.json' in files
        status = "✓" if (has_weights and has_config) else "✗"
        print(f"  {status}  {folder}")

  ✓  model1_sentfin
  ✓  model1_sentfin_v2
  ✓  model1_sentfin_v3
  ✓  model2_indian
  ✓  model2_indian_v2
  ✓  model2_indian_v3
  ✓  model3_combined
  ✓  model3_combined_v2
  ✗  model3_expA
  ✓  model3_expA_fixed
  ✓  model3_expB
  ✓  model3_expC


In [11]:
trainer_m3_a = train_model_v2(
    model_name    = "M3 — Exp A (lower lr + more epochs)",
    train_dataset = ds_m3_train,
    val_dataset   = ds_m3_val,
    output_dir    = f"{MDIR}model3_expA",
    learning_rate = 1e-5,    # halved from 2e-5
    dropout       = 0.2,     # same
    freeze_layers = 4,       # same
    weight_decay  = 0.1,     # same
    epochs        = 15       # more room to converge
)

NameError: name 'train_model_v2' is not defined

In [24]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

def train_model_llrd(model_name, train_dataset, val_dataset,
                     output_dir, epochs=10,
                     base_lr=2e-5, decay=0.9,
                     dropout=0.2, weight_decay=0.1):

    print(f"\n{'█'*55}")
    print(f"  Training  : {model_name}")
    print(f"  Base LR   : {base_lr}  Decay: {decay}")
    print(f"  Dropout   : {dropout}")
    print(f"{'█'*55}\n")

    # Load model with dropout
    config = AutoConfig.from_pretrained(
        "ProsusAI/finbert",
        num_labels                   = 3,
        id2label                     = ID2LABEL,
        label2id                     = LABEL2ID,
        hidden_dropout_prob          = dropout,
        attention_probs_dropout_prob = dropout,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        "ProsusAI/finbert",
        config                  = config,
        ignore_mismatched_sizes = True
    ).to(device)

    # LLRD — each layer gets decayed learning rate
    # Layer 11 (top)    → base_lr
    # Layer 10          → base_lr × decay
    # Layer 9           → base_lr × decay²
    # ...
    # Layer 0 (bottom)  → base_lr × decay¹¹
    # Embeddings        → base_lr × decay¹²
    optimizer_params = []

    # Classifier head — full learning rate
    optimizer_params.append({
        "params"      : [p for n,p in model.named_parameters()
                         if "classifier" in n],
        "lr"          : base_lr,
        "weight_decay": weight_decay
    })

    # BERT encoder layers — decay from top to bottom
    for layer_num in range(11, -1, -1):
        layer_lr = base_lr * (decay ** (11 - layer_num))
        optimizer_params.append({
            "params": [p for n,p in model.named_parameters()
                      if f"encoder.layer.{layer_num}." in n],
            "lr"    : layer_lr,
            "weight_decay": weight_decay
        })
        print(f"  Layer {layer_num:>2} lr: {layer_lr:.2e}")

    # Embeddings — smallest lr
    optimizer_params.append({
        "params"      : [p for n,p in model.named_parameters()
                         if "embeddings" in n],
        "lr"          : base_lr * (decay ** 12),
        "weight_decay": weight_decay
    })

    optimizer = AdamW(optimizer_params)

    # Training arguments — no lr here since optimizer handles it
    args = TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = epochs,
        per_device_train_batch_size = 32,
        per_device_eval_batch_size  = 64,
        weight_decay                = 0,      # handled in optimizer
        warmup_steps                = 200,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "f1",
        greater_is_better           = True,
        logging_steps               = 50,
        report_to                   = "none",
        seed                        = 42,
        fp16                        = True,
    )

    trainer = Trainer(
        model           = model,
        args            = args,
        train_dataset   = train_dataset,
        eval_dataset    = val_dataset,
        compute_metrics = compute_metrics,
        optimizers      = (optimizer, None),  # pass custom optimizer
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)]
    )

    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"\n  ✓ Saved → {output_dir}")
    return trainer


# Run Experiment C
trainer_m3_c = train_model_llrd(
    model_name    = "M3 — Exp C (LLRD)",
    train_dataset = ds_m3_train,
    val_dataset   = ds_m3_val,
    output_dir    = f"{MDIR}model3_expC",
    base_lr       = 2e-5,     # 2e-5 changed
    decay         = 0.9,     # each layer gets 90% of layer above's lr
    dropout       = 0.2,
    weight_decay  = 0.1,
    epochs        = 15    #earlier 10
)


███████████████████████████████████████████████████████
  Training  : M3 — Exp C (LLRD)
  Base LR   : 2e-05  Decay: 0.9
  Dropout   : 0.2
███████████████████████████████████████████████████████



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Layer 11 lr: 2.00e-05
  Layer 10 lr: 1.80e-05
  Layer  9 lr: 1.62e-05
  Layer  8 lr: 1.46e-05
  Layer  7 lr: 1.31e-05
  Layer  6 lr: 1.18e-05
  Layer  5 lr: 1.06e-05
  Layer  4 lr: 9.57e-06
  Layer  3 lr: 8.61e-06
  Layer  2 lr: 7.75e-06
  Layer  1 lr: 6.97e-06
  Layer  0 lr: 6.28e-06


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.583915,0.517567,0.791300,0.792300
2,0.395580,0.412484,0.840000,0.841500
3,0.348419,0.388613,0.853400,0.853200
4,0.312244,0.406130,0.851400,0.852300
5,0.258312,0.397160,0.858600,0.858500
6,0.225320,0.424304,0.867800,0.867500
7,0.200692,0.444879,0.858000,0.858500
8,0.147352,0.471727,0.860400,0.860300
9,0.143570,0.477792,0.867900,0.867500
10,0.130672,0.481741,0.863700,0.863900


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  ✓ Saved → /content/drive/MyDrive/finbert_project/models/model3_expC


In [19]:
# ══════════════════════════════════════════════════════════════════
# TRAINING ARGUMENTS — updated to use F1 as stopping metric
# ══════════════════════════════════════════════════════════════════
def get_training_args(output_dir, epochs=15, learning_rate=2e-5):
    return TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = epochs,
        per_device_train_batch_size = 32,
        per_device_eval_batch_size  = 64,
        learning_rate               = learning_rate,
        weight_decay                = 0.1,
        warmup_steps                = 200,
        lr_scheduler_type           = "cosine",   # smooth lr decay
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "f1",       # ← F1 not eval_loss
        greater_is_better           = True,        # ← higher F1 = better
        logging_steps               = 50,
        report_to                   = "none",
        seed                        = 42,
        fp16                        = True,
    )


# ══════════════════════════════════════════════════════════════════
# TRAIN MODEL — same as before, learning_rate passed through
# ══════════════════════════════════════════════════════════════════
def train_model_v2(model_name, train_dataset, val_dataset,
                   output_dir, epochs=15,
                   learning_rate=2e-5,
                   dropout=0.2,
                   freeze_layers=4,
                   weight_decay=0.1):

    print(f"\n{'█'*55}")
    print(f"  Training  : {model_name}")
    print(f"  LR        : {learning_rate}")
    print(f"  Dropout   : {dropout}")
    print(f"  Frozen    : bottom {freeze_layers} layers")
    print(f"  WD        : {weight_decay}")
    print(f"  Epochs    : {epochs}")
    print(f"{'█'*55}\n")

    # Load model with custom dropout
    config = AutoConfig.from_pretrained(
        "ProsusAI/finbert",
        num_labels                   = 3,
        id2label                     = ID2LABEL,
        label2id                     = LABEL2ID,
        hidden_dropout_prob          = dropout,
        attention_probs_dropout_prob = dropout,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        "ProsusAI/finbert",
        config                  = config,
        ignore_mismatched_sizes = True
    )

    # Freeze bottom layers
    for param in model.bert.embeddings.parameters():
        param.requires_grad = False
    for i in range(freeze_layers):
        for param in model.bert.encoder.layer[i].parameters():
            param.requires_grad = False

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"  Trainable params : {trainable:,}")
    print(f"  Frozen params    : {frozen:,}")

    trainer = Trainer(
        model           = model,
        args            = get_training_args(output_dir, epochs, learning_rate),
        train_dataset   = train_dataset,
        eval_dataset    = val_dataset,
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)]
    )

    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"\n  ✓ Saved → {output_dir}")

    return trainer


# ══════════════════════════════════════════════════════════════════
# RUN EXP A — fixed with F1 stopping metric
# ══════════════════════════════════════════════════════════════════
trainer_m3_a = train_model_v2(
    model_name    = "M3 — Exp A fixed (F1 stopping)",
    train_dataset = ds_m3_train,
    val_dataset   = ds_m3_val,
    output_dir    = f"{MDIR}model3_expA_fixed",
    learning_rate = 1e-5,
    dropout       = 0.2,
    freeze_layers = 4,
    weight_decay  = 0.1,
    epochs        = 15
)


███████████████████████████████████████████████████████
  Training  : M3 — Exp A fixed (F1 stopping)
  LR        : 1e-05
  Dropout   : 0.2
  Frozen    : bottom 4 layers
  WD        : 0.1
  Epochs    : 15
███████████████████████████████████████████████████████



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Trainable params : 57,295,875
  Frozen params    : 52,188,672


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.692488,0.582092,0.767700,0.768100
2,0.452071,0.472349,0.816800,0.817400
3,0.417850,0.448307,0.832500,0.831700
4,0.381024,0.436124,0.839100,0.839700
5,0.337073,0.410134,0.843000,0.842400
6,0.328522,0.416914,0.858300,0.857700
7,0.301730,0.415686,0.853600,0.854100
8,0.246227,0.415666,0.861600,0.861200
9,0.248930,0.416780,0.866100,0.865700
10,0.218005,0.421181,0.859200,0.859400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  ✓ Saved → /content/drive/MyDrive/finbert_project/models/model3_expA_fixed


In [20]:
trainer_m3_b = train_model_v2(
    model_name    = "M3 — Exp B (more trainable layers)",
    train_dataset = ds_m3_train,
    val_dataset   = ds_m3_val,
    output_dir    = f"{MDIR}model3_expB",
    learning_rate = 2e-5,    # same
    dropout       = 0.15,    # slightly less dropout
    freeze_layers = 2,       # unfreeze more — 8,936 rows can handle it
    weight_decay  = 0.1,     # same
    epochs        = 10
)


███████████████████████████████████████████████████████
  Training  : M3 — Exp B (more trainable layers)
  LR        : 2e-05
  Dropout   : 0.15
  Frozen    : bottom 2 layers
  WD        : 0.1
  Epochs    : 10
███████████████████████████████████████████████████████



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Trainable params : 71,471,619
  Frozen params    : 38,012,928


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.522273,0.472033,0.814900,0.814700
2,0.351603,0.374216,0.855500,0.856800
3,0.289768,0.373119,0.861100,0.861200
4,0.214445,0.408800,0.848200,0.848700
5,0.148258,0.466202,0.860200,0.860300
6,0.101037,0.491363,0.868000,0.867500
7,0.098572,0.545014,0.857300,0.857700
8,0.054777,0.584718,0.857600,0.857700
9,0.078994,0.597298,0.858200,0.858500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  ✓ Saved → /content/drive/MyDrive/finbert_project/models/model3_expB


In [13]:
# ══════════════════════════════════════════════════════════════════
# STEP 1 — EVALUATE ONE MODEL ON ONE TEST SET
# Returns predictions, true labels, and all metrics
# ══════════════════════════════════════════════════════════════════
def evaluate_model(model_path, test_dataset, test_df, model_label):

    print(f"\n  Loading {model_label} from {model_path}...")

    # Load saved model and tokenizer
    model     = AutoModelForSequenceClassification.from_pretrained(model_path)
    model     = model.to(device)
    model.eval()   # evaluation mode — disables dropout

    # DataLoader — batches test data for efficient inference
    loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    all_preds  = []
    all_labels = []

    # No gradient computation needed during evaluation — saves memory
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            # Forward pass — get raw logits
            outputs = model(input_ids=input_ids,
                           attention_mask=attention_mask)

            # Take highest logit as prediction
            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Convert to arrays
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Compute metrics
    f1  = f1_score(all_labels, all_preds, average='weighted')
    acc = accuracy_score(all_labels, all_preds)

    print(f"\n  {'─'*50}")
    print(f"  Model  : {model_label}")
    print(f"  {'─'*50}")
    print(f"  Weighted F1 : {f1:.4f}")
    print(f"  Accuracy    : {acc:.4f}")
    print(f"\n  Per-class report:")
    print(classification_report(
        all_labels, all_preds,
        target_names = [ID2LABEL[i] for i in range(3)],
        digits       = 4
    ))

    return {
        "model"      : model_label,
        "f1"         : round(f1,  4),
        "accuracy"   : round(acc, 4),
        "preds"      : all_preds,
        "labels"     : all_labels,
    }

    # Quick comparison of all M3 versions on test sets
versions = {
    "M3 V2"       : f"{MDIR}model3_combined_v2",
    "M3 ExpA fix" : f"{MDIR}model3_expA_fixed",
    "M3 ExpB"     : f"{MDIR}model3_expB",
    "M3 ExpC"     : f"{MDIR}model3_expC",
}

print("="*65)
print("  ALL M3 VERSIONS — TEST SET COMPARISON")
print("="*65)
print(f"  {'Version':<15} {'SF F1':>8} {'IN F1':>8} {'SF Acc':>8} {'IN Acc':>8}")
print(f"  {'─'*55}")

best_version = None
best_avg_f1  = 0

for name, path in versions.items():
    sf = evaluate_model(path, ds_test_sf, test_sf, name)
    in_ = evaluate_model(path, ds_test_in, test_in, name)
    avg_f1 = (sf['f1'] + in_['f1']) / 2

    print(f"  {name:<15} {sf['f1']:>8} {in_['f1']:>8} "
          f"{sf['accuracy']:>8} {in_['accuracy']:>8}  avg={avg_f1:.4f}")

    if avg_f1 > best_avg_f1:
        best_avg_f1  = avg_f1
        best_version = name

print(f"  {'─'*55}")
print(f"  Best version : {best_version} (avg F1={best_avg_f1:.4f})")
print("="*65)

  ALL M3 VERSIONS — TEST SET COMPARISON
  Version            SF F1    IN F1   SF Acc   IN Acc
  ───────────────────────────────────────────────────────

  Loading M3 V2 from /content/drive/MyDrive/finbert_project/models/model3_combined_v2...


Loading weights:   0%|          | 0/201 [00:05<?, ?it/s]


  ──────────────────────────────────────────────────
  Model  : M3 V2
  ──────────────────────────────────────────────────
  Weighted F1 : 0.8859
  Accuracy    : 0.8854

  Per-class report:
              precision    recall  f1-score   support

    negative     0.9216    0.8744    0.8974       215
     neutral     0.8258    0.8790    0.8516       248
    positive     0.9170    0.9000    0.9084       270

    accuracy                         0.8854       733
   macro avg     0.8881    0.8845    0.8858       733
weighted avg     0.8875    0.8854    0.8859       733


  Loading M3 V2 from /content/drive/MyDrive/finbert_project/models/model3_combined_v2...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


  ──────────────────────────────────────────────────
  Model  : M3 V2
  ──────────────────────────────────────────────────
  Weighted F1 : 0.8521
  Accuracy    : 0.8519

  Per-class report:
              precision    recall  f1-score   support

    negative     0.8556    0.8556    0.8556        90
     neutral     0.8030    0.8092    0.8061       131
    positive     0.8896    0.8841    0.8869       164

    accuracy                         0.8519       385
   macro avg     0.8494    0.8496    0.8495       385
weighted avg     0.8522    0.8519    0.8521       385

  M3 V2             0.8859   0.8521   0.8854   0.8519  avg=0.8690

  Loading M3 ExpA fix from /content/drive/MyDrive/finbert_project/models/model3_expA_fixed...


...


  ──────────────────────────────────────────────────
  Model  : M3 ExpB
  ──────────────────────────────────────────────────
  Weighted F1 : 0.8501
  Accuracy    : 0.8494

  Per-class report:
              precision    recall  f1-score   support

    negative     0.8652    0.8556    0.8603        90
     neutral     0.7857    0.8397    0.8118       131
    positive     0.8974    0.8537    0.8750       164

    accuracy                         0.8494       385
   macro avg     0.8494    0.8496    0.8490       385
weighted avg     0.8519    0.8494    0.8501       385

  M3 ExpB           0.8808   0.8501   0.8799   0.8494  avg=0.8655

  Loading M3 ExpC from /content/drive/MyDrive/finbert_project/models/model3_expC...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


  ──────────────────────────────────────────────────
  Model  : M3 ExpC
  ──────────────────────────────────────────────────
  Weighted F1 : 0.8843
  Accuracy    : 0.8840

  Per-class report:
              precision    recall  f1-score   support

    negative     0.9246    0.8558    0.8889       215
     neutral     0.8321    0.8790    0.8549       248
    positive     0.9044    0.9111    0.9077       270

    accuracy                         0.8840       733
   macro avg     0.8870    0.8820    0.8838       733
weighted avg     0.8859    0.8840    0.8843       733


  Loading M3 ExpC from /content/drive/MyDrive/finbert_project/models/model3_expC...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


  ──────────────────────────────────────────────────
  Model  : M3 ExpC
  ──────────────────────────────────────────────────
  Weighted F1 : 0.8651
  Accuracy    : 0.8649

  Per-class report:
              precision    recall  f1-score   support

    negative     0.9070    0.8667    0.8864        90
     neutral     0.8120    0.8244    0.8182       131
    positive     0.8855    0.8963    0.8909       164

    accuracy                         0.8649       385
   macro avg     0.8682    0.8625    0.8652       385
weighted avg     0.8655    0.8649    0.8651       385

  M3 ExpC           0.8843   0.8651    0.884   0.8649  avg=0.8747
  ───────────────────────────────────────────────────────
  Best version : M3 ExpA fix (avg F1=0.8756)


In [12]:
# ══════════════════════════════════════════════════════════════════
# UPDATED get_training_args — adds label_smoothing parameter
# ══════════════════════════════════════════════════════════════════
def get_training_args(output_dir, epochs=10, learning_rate=2e-5,
                      weight_decay=0.1, label_smoothing=0.0):
    return TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = epochs,
        per_device_train_batch_size = 32,
        per_device_eval_batch_size  = 64,
        learning_rate               = learning_rate,
        weight_decay                = weight_decay,
        warmup_steps                = 200,
        lr_scheduler_type           = "cosine",
        label_smoothing_factor      = label_smoothing,  # ← new
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "f1",
        greater_is_better           = True,
        logging_steps               = 50,
        report_to                   = "none",
        seed                        = 42,
        fp16                        = True,
    )


# ══════════════════════════════════════════════════════════════════
# UPDATED train_model — adds label_smoothing parameter
# ══════════════════════════════════════════════════════════════════
def train_model(model_name, train_dataset, val_dataset,
                output_dir, epochs=10, learning_rate=2e-5,
                dropout=0.2, freeze_layers=4,
                weight_decay=0.1, label_smoothing=0.0):

    print(f"\n{'█'*55}")
    print(f"  Training      : {model_name}")
    print(f"  Train samples : {len(train_dataset)}")
    print(f"  Val samples   : {len(val_dataset)}")
    print(f"  LR            : {learning_rate}")
    print(f"  Dropout       : {dropout}")
    print(f"  Frozen layers : {freeze_layers}")
    print(f"  Weight decay  : {weight_decay}")
    print(f"  Label smooth  : {label_smoothing}")
    print(f"  Epochs (max)  : {epochs}")
    print(f"{'█'*55}\n")

    config = AutoConfig.from_pretrained(
        "ProsusAI/finbert",
        num_labels                   = 3,
        id2label                     = ID2LABEL,
        label2id                     = LABEL2ID,
        hidden_dropout_prob          = dropout,
        attention_probs_dropout_prob = dropout,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        "ProsusAI/finbert",
        config                  = config,
        ignore_mismatched_sizes = True
    )

    # Freeze bottom layers
    for param in model.bert.embeddings.parameters():
        param.requires_grad = False
    for i in range(freeze_layers):
        for param in model.bert.encoder.layer[i].parameters():
            param.requires_grad = False

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"  Trainable : {trainable:,}")
    print(f"  Frozen    : {frozen:,}")

    trainer = Trainer(
        model           = model,
        args            = get_training_args(
                            output_dir, epochs, learning_rate,
                            weight_decay, label_smoothing
                          ),
        train_dataset   = train_dataset,
        eval_dataset    = val_dataset,
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)]
    )

    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"\n  ✓ Saved → {output_dir}")
    return trainer




In [13]:
# ══════════════════════════════════════════════════════════════════
# RUN ALL 3 M1 EXPERIMENTS
# ══════════════════════════════════════════════════════════════════

# Experiment 1 — Label smoothing (same layers as V3)
trainer_m1_exp1 = train_model(
    model_name      = "M1 — Exp1 (label smoothing)",
    train_dataset   = ds_m1_train,
    val_dataset     = ds_m1_val,
    output_dir      = f"{MDIR}model1_exp1",
    learning_rate   = 1e-5,
    dropout         = 0.2,
    freeze_layers   = 3,
    weight_decay    = 0.05,
    label_smoothing = 0.1,
    epochs          = 10
)


███████████████████████████████████████████████████████
  Training      : M1 — Exp1 (label smoothing)
  Train samples : 5861
  Val samples   : 733
  LR            : 1e-05
  Dropout       : 0.2
  Frozen layers : 3
  Weight decay  : 0.05
  Label smooth  : 0.1
  Epochs (max)  : 10
███████████████████████████████████████████████████████



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Trainable : 64,383,747
  Frozen    : 45,100,800


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,1.015985,0.906141,0.585900,0.593500
2,0.708968,0.661338,0.797200,0.796700
3,0.587221,0.605494,0.825400,0.825400
4,0.564170,0.583133,0.825700,0.825400
5,0.545186,0.567139,0.837000,0.836300
6,0.518102,0.567400,0.838900,0.839000
7,0.469029,0.568409,0.845800,0.845800
8,0.475039,0.569655,0.846000,0.845800
9,0.482230,0.568001,0.848700,0.848600
10,0.455884,0.568330,0.850100,0.849900


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  ✓ Saved → /content/drive/MyDrive/finbert_project/models/model1_exp1


In [14]:
# Experiment 2 — More layers + label smoothing
trainer_m1_exp2 = train_model(
    model_name      = "M1 — Exp2 (more layers + smoothing)",
    train_dataset   = ds_m1_train,
    val_dataset     = ds_m1_val,
    output_dir      = f"{MDIR}model1_exp2",
    learning_rate   = 2e-5,
    dropout         = 0.2,
    freeze_layers   = 2,
    weight_decay    = 0.1,
    label_smoothing = 0.1,
    epochs          = 10
)


███████████████████████████████████████████████████████
  Training      : M1 — Exp2 (more layers + smoothing)
  Train samples : 5861
  Val samples   : 733
  LR            : 2e-05
  Dropout       : 0.2
  Frozen layers : 2
  Weight decay  : 0.1
  Label smooth  : 0.1
  Epochs (max)  : 10
███████████████████████████████████████████████████████



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Trainable : 71,471,619
  Frozen    : 38,012,928


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.938276,0.785716,0.700700,0.701200
2,0.631199,0.607133,0.826800,0.826700
3,0.526235,0.560576,0.849400,0.849900
4,0.483492,0.563153,0.846700,0.847200
5,0.457958,0.555644,0.862300,0.862200
6,0.424997,0.573595,0.859500,0.859500
7,0.394423,0.579192,0.870300,0.870400
8,0.390812,0.589618,0.863900,0.863600
9,0.389693,0.589931,0.862300,0.862200
10,0.363649,0.591150,0.859700,0.859500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  ✓ Saved → /content/drive/MyDrive/finbert_project/models/model1_exp2


In [17]:
from transformers import TrainerCallback
from torch.optim import AdamW

# ══════════════════════════════════════════════════════════════════
# GRADUAL UNFREEZING CALLBACK
# Starts with all BERT layers frozen (only classifier trains)
# Unfreezes one layer at a time from top to bottom as training progresses
# ══════════════════════════════════════════════════════════════════
class GradualUnfreezingCallback(TrainerCallback):

    def __init__(self, model, total_steps, num_layers=12):
        self.model              = model
        self.total_steps        = total_steps
        self.num_layers         = num_layers
        self.steps_per_unfreeze = total_steps // (num_layers + 1)
        self.unfrozen_up_to     = -1
        print(f"  Gradual unfreezing every {self.steps_per_unfreeze} steps")

    def on_step_begin(self, args, state, control, **kwargs):
        current_step       = state.global_step
        layers_to_unfreeze = current_step // self.steps_per_unfreeze

        if layers_to_unfreeze > self.unfrozen_up_to:
            self.unfrozen_up_to = layers_to_unfreeze
            layer_idx           = self.num_layers - layers_to_unfreeze

            if 0 <= layer_idx < self.num_layers:
                for param in self.model.bert.encoder.layer[
                    layer_idx
                ].parameters():
                    param.requires_grad = True
                print(f"\n  ✓ Step {current_step} → "
                      f"unfroze layer {layer_idx}")

            elif layer_idx < 0:
                # All layers unfrozen — also unfreeze embeddings
                for param in self.model.bert.embeddings.parameters():
                    param.requires_grad = True
                print(f"\n  ✓ Step {current_step} → "
                      f"unfroze embeddings (all layers now active)")


# ══════════════════════════════════════════════════════════════════
# M3 — PAPER STYLE TRAINING
# Combines all 3 techniques from FinBERT paper:
#   1. Slanted Triangular LR (linear warmup + linear decay)
#   2. Gradual Unfreezing    (top → bottom, one layer at a time)
#   3. Discriminative FT     (LLRD: each layer gets decayed lr)
# Plus your proven improvements:
#   4. Higher dropout (0.2 vs paper's 0.1 — works better for Indian data)
#   5. Label smoothing (reduces overconfidence)
#   6. Larger batch size (32 vs paper's 64 — Colab GPU constraint)
# ══════════════════════════════════════════════════════════════════
def train_m3_paper_style(model_name, train_dataset, val_dataset,
                          output_dir, epochs=10,
                          base_lr=2e-5, dropout=0.2,
                          weight_decay=0.01, llrd_decay=0.85,
                          label_smoothing=0.1):

    print(f"\n{'█'*55}")
    print(f"  Training      : {model_name}")
    print(f"  Techniques    : Gradual Unfreeze + LLRD + STL")
    print(f"  Base LR       : {base_lr}")
    print(f"  LLRD decay    : {llrd_decay}")
    print(f"  Dropout       : {dropout}")
    print(f"  Label smooth  : {label_smoothing}")
    print(f"  Weight decay  : {weight_decay}")
    print(f"  Epochs (max)  : {epochs}")
    print(f"{'█'*55}\n")

    # ── Load model with custom dropout ────────────────────────────
    config = AutoConfig.from_pretrained(
        "ProsusAI/finbert",
        num_labels                   = 3,
        id2label                     = ID2LABEL,
        label2id                     = LABEL2ID,
        hidden_dropout_prob          = dropout,
        attention_probs_dropout_prob = dropout,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        "ProsusAI/finbert",
        config                  = config,
        ignore_mismatched_sizes = True
    )

    # ── Freeze ALL bert layers at start ───────────────────────────
    # Gradual unfreezing callback will unfreeze them progressively
    for param in model.bert.parameters():
        param.requires_grad = False

    trainable = sum(p.numel() for p in model.parameters()
                    if p.requires_grad)
    print(f"  Initial trainable : {trainable:,} params (classifier only)")

    # ── LLRD Optimizer ────────────────────────────────────────────
    # Discriminative fine-tuning — different lr per layer
    # Paper used discrimination rate 0.85
    # Layer 11 (top) → base_lr
    # Layer 10       → base_lr × 0.85
    # Layer 9        → base_lr × 0.85²
    # ...
    # Layer 0        → base_lr × 0.85¹¹
    # Embeddings     → base_lr × 0.85¹²
    optimizer_params = []

    # Classifier head — full learning rate
    optimizer_params.append({
        "params"      : [p for n, p in model.named_parameters()
                         if "classifier" in n],
        "lr"          : base_lr,
        "weight_decay": weight_decay
    })

    # BERT encoder layers — decay from top to bottom
    for layer_num in range(11, -1, -1):
        layer_lr = base_lr * (llrd_decay ** (11 - layer_num))
        optimizer_params.append({
            "params": [p for n, p in model.named_parameters()
                       if f"encoder.layer.{layer_num}." in n],
            "lr"          : layer_lr,
            "weight_decay": weight_decay
        })

    # Embeddings — smallest lr
    optimizer_params.append({
        "params"      : [p for n, p in model.named_parameters()
                         if "embeddings" in n],
        "lr"          : base_lr * (llrd_decay ** 12),
        "weight_decay": weight_decay
    })

    optimizer = AdamW(optimizer_params)

    # ── Compute total steps for gradual unfreezing ────────────────
    steps_per_epoch = len(train_dataset) // 32
    total_steps     = steps_per_epoch * epochs
    warmup_steps    = int(total_steps * 0.1)   # 10% warmup (paper: 0.2)
    print(f"  Total steps   : {total_steps}")
    print(f"  Warmup steps  : {warmup_steps}")
    print(f"  Steps/unfreeze: {total_steps // 13}")

    # ── Training arguments ────────────────────────────────────────
    args = TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = epochs,

        # Batch sizes
        per_device_train_batch_size = 32,
        per_device_eval_batch_size  = 64,

        # LR handled by optimizer — set dummy value here
        learning_rate               = base_lr,
        weight_decay                = 0,        # handled in optimizer

        # Slanted triangular LR = linear warmup + linear decay
        warmup_steps                = warmup_steps,
        lr_scheduler_type           = "linear", # paper used linear

        # Label smoothing
        label_smoothing_factor      = label_smoothing,

        # Evaluation
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "f1",
        greater_is_better           = True,

        # Logging
        logging_steps               = 50,
        report_to                   = "none",

        # Reproducibility
        seed                        = 42,
        fp16                        = True,
    )

    # ── Callbacks ─────────────────────────────────────────────────
    gu_callback = GradualUnfreezingCallback(
        model       = model,
        total_steps = total_steps,
        num_layers  = 12
    )

    trainer = Trainer(
        model           = model,
        args            = args,
        train_dataset   = train_dataset,
        eval_dataset    = val_dataset,
        compute_metrics = compute_metrics,
        optimizers      = (optimizer, None),   # custom LLRD optimizer
        callbacks       = [
            gu_callback,
            EarlyStoppingCallback(early_stopping_patience=3)
        ]
    )

    # ── Train ─────────────────────────────────────────────────────
    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"\n  ✓ Saved → {output_dir}")
    return trainer


# ══════════════════════════════════════════════════════════════════
# RUN M3 — PAPER STYLE
# ══════════════════════════════════════════════════════════════════
trainer_m3_paper = train_m3_paper_style(
    model_name      = "M3 — Paper Style (GU + LLRD + STL)",
    train_dataset   = ds_m3_train,
    val_dataset     = ds_m3_val,
    output_dir      = f"{MDIR}model3_paper_style",
    epochs          = 10,
    base_lr         = 2e-5,       # paper's exact lr
    dropout         = 0.2,        # your proven setting
    weight_decay    = 0.01,       # paper's exact wd
    llrd_decay      = 0.85,       # paper's exact discrimination rate
    label_smoothing = 0.1         # your addition — paper didn't use this
)


███████████████████████████████████████████████████████
  Training      : M3 — Paper Style (GU + LLRD + STL)
  Techniques    : Gradual Unfreeze + LLRD + STL
  Base LR       : 2e-05
  LLRD decay    : 0.85
  Dropout       : 0.2
  Label smooth  : 0.1
  Weight decay  : 0.01
  Epochs (max)  : 10
███████████████████████████████████████████████████████



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Initial trainable : 2,307 params (classifier only)
  Total steps   : 2790
  Warmup steps  : 279
  Steps/unfreeze: 214
  Gradual unfreezing every 214 steps


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,1.617562,0.883031,0.672300,0.675900
2,0.688063,0.695036,0.772600,0.772600
3,0.683189,0.664925,0.793000,0.792300
4,0.649394,0.649512,0.800300,0.801300
5,0.619117,0.628492,0.817000,0.816500
6,0.616405,0.617941,0.823300,0.822700
7,0.595428,0.613140,0.823900,0.823600
8,0.540779,0.606370,0.823400,0.822700
9,0.549673,0.599811,0.835400,0.835300
10,0.544614,0.598899,0.834800,0.834400



  ✓ Step 214 → unfroze layer 11


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  ✓ Step 2568 → unfroze layer 0

  ✓ Step 2782 → unfroze embeddings (all layers now active)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  ✓ Saved → /content/drive/MyDrive/finbert_project/models/model3_paper_style


In [22]:
# ══════════════════════════════════════════════════════════════════
# COMPLETE EVALUATION — ALL MODELS ON BOTH TEST SETS
# ══════════════════════════════════════════════════════════════════

# ── Helper: load model safely ─────────────────────────────────────
def load_model_safe(model_path):
    """
    Handles both V1-style and V2/V3-style saved models.
    V2/V3 models saved with AutoConfig may have missing model_type
    in config.json — this function handles both cases.
    """
    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            model_path,
            ignore_mismatched_sizes = True
        )
    except ValueError:
        config = AutoConfig.from_pretrained(
            "ProsusAI/finbert",
            num_labels = 3,
            id2label   = ID2LABEL,
            label2id   = LABEL2ID,
        )
        model = AutoModelForSequenceClassification.from_pretrained(
            model_path,
            config                  = config,
            ignore_mismatched_sizes = True
        )
    return model.to(device)


# ── Core evaluation function ──────────────────────────────────────
def evaluate_model(model_path, test_dataset, model_label):
    model = load_model_safe(model_path)
    model.eval()

    loader     = DataLoader(test_dataset, batch_size=32, shuffle=False)
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            outputs        = model(input_ids      = input_ids,
                                   attention_mask = attention_mask)
            preds          = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    f1  = f1_score(all_labels, all_preds, average='weighted')
    acc = accuracy_score(all_labels, all_preds)

    return {
        "model"   : model_label,
        "f1"      : round(f1,  4),
        "accuracy": round(acc, 4),
        "preds"   : all_preds,
        "labels"  : all_labels,
    }


# ── Vanilla FinBERT evaluation ────────────────────────────────────
def evaluate_vanilla(test_dataset, model_label):
    model = AutoModelForSequenceClassification.from_pretrained(
        "ProsusAI/finbert"
    ).to(device)
    model.eval()

    # Vanilla FinBERT's original label mapping — must remap
    VANILLA_ID2LABEL = {0: "positive", 1: "negative", 2: "neutral"}

    loader     = DataLoader(test_dataset, batch_size=32, shuffle=False)
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            outputs        = model(input_ids      = input_ids,
                                   attention_mask = attention_mask)
            preds          = torch.argmax(outputs.logits, dim=1)

            # Remap vanilla predictions to your label scheme
            remapped = [
                LABEL2ID[VANILLA_ID2LABEL[p]]
                for p in preds.cpu().numpy()
            ]
            all_preds.extend(remapped)
            all_labels.extend(labels.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    f1  = f1_score(all_labels, all_preds, average='weighted')
    acc = accuracy_score(all_labels, all_preds)

    return {
        "model"   : model_label,
        "f1"      : round(f1,  4),
        "accuracy": round(acc, 4),
        "preds"   : all_preds,
        "labels"  : all_labels,
    }


# ══════════════════════════════════════════════════════════════════
# ALL MODELS TO EVALUATE
# Add or remove paths as needed based on what's saved in Drive
# ══════════════════════════════════════════════════════════════════
models_to_eval = {
    # ── Baseline ──────────────────────────────────────────────────
    "Vanilla FinBERT"    : "vanilla",

    # ── M1 versions ───────────────────────────────────────────────
    "M1 V1"              : f"{MDIR}model1_sentfin",
    "M1 Exp1 (smooth)"   : f"{MDIR}model1_exp1",
    "M1 Exp2 (layers)"   : f"{MDIR}model1_exp2",
    "M1 Exp3 (LLRD)"     : f"{MDIR}model1_exp3",
    "M1 ExpA (LLRD)"     : f"{MDIR}model1_expA",

    # ── M2 versions ───────────────────────────────────────────────
    "M2 V1"              : f"{MDIR}model2_indian",
    "M2 V3"              : f"{MDIR}model2_indian_v3",
    "M2 ExpB (LLRD)"     : f"{MDIR}model2_expB",


    # ── M3 versions ───────────────────────────────────────────────
    "M3 ExpA fixed"      : f"{MDIR}model3_expA_fixed",
    "M3 ExpB"            : f"{MDIR}model3_expB",
    "M3 ExpC (LLRD)"     : f"{MDIR}model3_expC",
    "M3 Paper Style"     : f"{MDIR}model3_paper_style",
}


# ══════════════════════════════════════════════════════════════════
# RUN EVALUATION — ALL MODELS ON BOTH TEST SETS
# ══════════════════════════════════════════════════════════════════
print("█"*65)
print("  FULL EVALUATION — ALL MODELS × BOTH TEST SETS")
print("█"*65)

sf_results = {}
in_results = {}

for name, path in models_to_eval.items():
    print(f"\n  Evaluating: {name}")
    print(f"  {'─'*50}")

    try:
        if path == "vanilla":
            sf_results[name] = evaluate_vanilla(ds_test_sf, name)
            in_results[name] = evaluate_vanilla(ds_test_in, name)
        else:
            sf_results[name] = evaluate_model(path, ds_test_sf, name)
            in_results[name] = evaluate_model(path, ds_test_in, name)

        print(f"  SF  F1={sf_results[name]['f1']:.4f}  "
              f"Acc={sf_results[name]['accuracy']:.4f}")
        print(f"  IN  F1={in_results[name]['f1']:.4f}  "
              f"Acc={in_results[name]['accuracy']:.4f}")

    except Exception as e:
        print(f"  ✗ Failed to load — {e}")
        sf_results[name] = {"f1": None, "accuracy": None,
                             "preds": None, "labels": None}
        in_results[name] = {"f1": None, "accuracy": None,
                             "preds": None, "labels": None}


# ══════════════════════════════════════════════════════════════════
# RESULTS TABLE
# ══════════════════════════════════════════════════════════════════
rows = []
for name in models_to_eval:
    sf  = sf_results[name]
    in_ = in_results[name]
    if sf["f1"] is not None:
        avg = round((sf["f1"] + in_["f1"]) / 2, 4)
        rows.append({
            "Model"      : name,
            "SF F1"      : sf["f1"],
            "IN F1"      : in_["f1"],
            "Avg F1"     : avg,
            "SF Acc"     : sf["accuracy"],
            "IN Acc"     : in_["accuracy"],
        })
    else:
        rows.append({
            "Model"      : name,
            "SF F1"      : "—",
            "IN F1"      : "—",
            "Avg F1"     : "—",
            "SF Acc"     : "—",
            "IN Acc"     : "—",
        })

results_df = pd.DataFrame(rows)

print("\n" + "═"*75)
print("  FINAL RESULTS TABLE")
print("═"*75)
print(results_df.to_string(index=False))
print("═"*75)

# Find best model by avg F1
numeric = results_df[results_df["Avg F1"] != "—"].copy()
numeric["Avg F1"] = numeric["Avg F1"].astype(float)
best_row = numeric.loc[numeric["Avg F1"].idxmax()]
print(f"\n  ★ Best model : {best_row['Model']}")
print(f"    SF F1      : {best_row['SF F1']}")
print(f"    IN F1      : {best_row['IN F1']}")
print(f"    Avg F1     : {best_row['Avg F1']}")

# Save results
results_df.to_csv(f"{MDIR}full_evaluation_results.csv", index=False)
print(f"\n  Saved → {MDIR}full_evaluation_results.csv")


# ══════════════════════════════════════════════════════════════════
# PER-CLASS REPORT — BEST MODEL ONLY
# ══════════════════════════════════════════════════════════════════
best_name = best_row['Model']
print(f"\n{'═'*65}")
print(f"  PER-CLASS REPORT — {best_name}")
print(f"{'═'*65}")

print("\n  SEntFiN Test Set:")
print(classification_report(
    sf_results[best_name]["labels"],
    sf_results[best_name]["preds"],
    target_names = [ID2LABEL[i] for i in range(3)],
    digits       = 4
))

print("\n  Indian Test Set:")
print(classification_report(
    in_results[best_name]["labels"],
    in_results[best_name]["preds"],
    target_names = [ID2LABEL[i] for i in range(3)],
    digits       = 4
))


# ══════════════════════════════════════════════════════════════════
# SIGNIFICANCE TESTS — BEST MODEL vs EVERYTHING
# ══════════════════════════════════════════════════════════════════
from statsmodels.stats.contingency_tables import mcnemar as mc_test

def mcnemar_test(preds_a, preds_b, labels, name_a, name_b, test_name):
    correct_a = (np.array(preds_a) == np.array(labels))
    correct_b = (np.array(preds_b) == np.array(labels))
    b         = np.sum( correct_a & ~correct_b)
    c         = np.sum(~correct_a &  correct_b)
    table     = [[0, b], [c, 0]]
    result    = mc_test(table, exact=False, correction=True)
    sig       = "✓ significant" if result.pvalue < 0.05 else "✗ not significant"
    print(f"  {test_name} | {name_a} vs {name_b}")
    print(f"    b={b}  c={c}  p={result.pvalue:.4f}  {sig}")

def safe_mcnemar(sf_res, in_res, best_name, compare_name, label):
    """Run McNemar on both test sets only if the comparison model loaded."""
    if (compare_name not in sf_res
            or sf_res[compare_name]["preds"] is None):
        print(f"  ✗ Skipping {compare_name} — not loaded")
        return
    print(f"\n  vs {label}:")
    mcnemar_test(
        sf_res[best_name]["preds"],
        sf_res[compare_name]["preds"],
        sf_res[best_name]["labels"],
        best_name, compare_name, "SF test"
    )
    mcnemar_test(
        in_res[best_name]["preds"],
        in_res[compare_name]["preds"],
        in_res[best_name]["labels"],
        best_name, compare_name, "IN test"
    )

print(f"\n{'═'*65}")
print(f"  SIGNIFICANCE TESTS — {best_name} vs others")
print(f"{'═'*65}")

safe_mcnemar(sf_results, in_results, best_name, "Vanilla FinBERT",  "Vanilla FinBERT")
safe_mcnemar(sf_results, in_results, best_name, "M3 ExpA fixed",    "M3 ExpA fixed (previous best)")
safe_mcnemar(sf_results, in_results, best_name, "M1 ExpA (LLRD)",   "M1 ExpA — new LLRD")   # ← NEW
safe_mcnemar(sf_results, in_results, best_name, "M2 ExpB (LLRD)",   "M2 ExpB — new LLRD")   # ← NEW

print(f"\n{'═'*65}")
print("  EVALUATION COMPLETE")
print(f"{'═'*65}")


# Compare best M3 vs M3 ExpA (your previous best)
if "M3 ExpA fixed" in sf_results and sf_results["M3 ExpA fixed"]["preds"] is not None:
    print("\n  vs M3 ExpA fixed (previous best):")
    mcnemar_test(
        sf_results[best_name]["preds"],
        sf_results["M3 ExpA fixed"]["preds"],
        sf_results[best_name]["labels"],
        best_name, "M3 ExpA fixed", "SF test"
    )
    mcnemar_test(
        in_results[best_name]["preds"],
        in_results["M3 ExpA fixed"]["preds"],
        in_results[best_name]["labels"],
        best_name, "M3 ExpA fixed", "IN test"
    )

print(f"\n{'═'*65}")
print("  EVALUATION COMPLETE")
print(f"{'═'*65}")

█████████████████████████████████████████████████████████████████
  FULL EVALUATION — ALL MODELS × BOTH TEST SETS
█████████████████████████████████████████████████████████████████

  Evaluating: Vanilla FinBERT
  ──────────────────────────────────────────────────


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  SF  F1=0.7555  Acc=0.7531
  IN  F1=0.7824  Acc=0.7818

  Evaluating: M1 V1
  ──────────────────────────────────────────────────


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

  SF  F1=0.8843  Acc=0.8840
  IN  F1=0.8651  Acc=0.8649

  Evaluating: M3 Paper Style
  ──────────────────────────────────────────────────


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  SF  F1=0.8815  Acc=0.8813
  IN  F1=0.8287  Acc=0.8286

═══════════════════════════════════════════════════════════════════════════
  FINAL RESULTS TABLE
═══════════════════════════════════════════════════════════════════════════
           Model   SF F1   IN F1  Avg F1  SF Acc  IN Acc
 Vanilla FinBERT  0.7555  0.7824   0.769  0.7531  0.7818
           M1 V1  0.8734  0.8156  0.8445  0.8731  0.8156
M1 Exp1 (smooth)  0.8669  0.8062  0.8366  0.8663  0.8052
M1 Exp2 (layers)   0.878   0.825  0.8515  0.8772  0.8234
  M1 Exp3 (LLRD)       —       —       —       —       —
  M1 ExpA (LLRD)  0.8683  0.8011  0.8347  0.8677     0.8
           M2 V1  0.7683  0.8671  0.8177  0.7694  0.8675
           M2 V3   0.759  0.8564  0.8077  0.7599  0.8571
  M2 ExpB (LLRD)    0.76  0.8594  0.8097  0.7613  0.8597
   M3 ExpA fixed  0.8886  0.8625  0.8756  0.8881  0.8623
         M3 ExpB  0.8808  0.8501  0.8655  0.8799  0.8494
  M3 ExpC (LLRD)  0.8843  0.8651  0.8747   0.884  0.8649
  M3 Paper Style  0.8815  0.

/usr/local/lib/python3.12/dist-packages/statsmodels/stats/contingency_tables.py:1348: RuntimeWarning: divide by zero encountered in scalar divide
  statistic = (np.abs(n1 - n2) - corr)**2 / (1. * (n1 + n2))


In [19]:
from statsmodels.stats.contingency_tables import mcnemar as mc_test

print("="*65)
print("  CORRECTED SIGNIFICANCE TESTS")
print("="*65)

def mcnemar_test(preds_a, preds_b, labels, name_a, name_b, test_name):
    correct_a = (np.array(preds_a) == np.array(labels))
    correct_b = (np.array(preds_b) == np.array(labels))
    b         = np.sum( correct_a & ~correct_b)
    c         = np.sum(~correct_a &  correct_b)

    if b + c == 0:
        print(f"  {test_name} | {name_a} vs {name_b}")
        print(f"    b=0  c=0  → identical predictions, test not applicable")
        return

    table  = [[0, b], [c, 0]]
    result = mc_test(table, exact=False, correction=True)
    sig    = "✓ significant" if result.pvalue < 0.05 else "✗ not significant"
    print(f"  {test_name} | {name_a} vs {name_b}")
    print(f"    b={b}  c={c}  p={result.pvalue:.4f}  {sig}")

# ── Best M3 vs Vanilla ────────────────────────────────────────────
print("\n  M3 ExpA fixed vs Vanilla FinBERT:")
mcnemar_test(
    sf_results["M3 ExpA fixed"]["preds"],
    sf_results["Vanilla FinBERT"]["preds"],
    sf_results["M3 ExpA fixed"]["labels"],
    "M3 ExpA", "Vanilla", "SF test"
)
mcnemar_test(
    in_results["M3 ExpA fixed"]["preds"],
    in_results["Vanilla FinBERT"]["preds"],
    in_results["M3 ExpA fixed"]["labels"],
    "M3 ExpA", "Vanilla", "IN test"
)

# ── Best M3 vs Best M1 ────────────────────────────────────────────
print("\n  M3 ExpA fixed vs M1 Exp2 (best M1):")
mcnemar_test(
    sf_results["M3 ExpA fixed"]["preds"],
    sf_results["M1 Exp2 (layers)"]["preds"],
    sf_results["M3 ExpA fixed"]["labels"],
    "M3 ExpA", "M1 Exp2", "SF test"
)
mcnemar_test(
    in_results["M3 ExpA fixed"]["preds"],
    in_results["M1 Exp2 (layers)"]["preds"],
    in_results["M3 ExpA fixed"]["labels"],
    "M3 ExpA", "M1 Exp2", "IN test"
)

# ── Best M3 vs Best M2 ────────────────────────────────────────────
print("\n  M3 ExpA fixed vs M2 V1 (best M2):")
mcnemar_test(
    sf_results["M3 ExpA fixed"]["preds"],
    sf_results["M2 V1"]["preds"],
    sf_results["M3 ExpA fixed"]["labels"],
    "M3 ExpA", "M2 V1", "SF test"
)
mcnemar_test(
    in_results["M3 ExpA fixed"]["preds"],
    in_results["M2 V1"]["preds"],
    in_results["M3 ExpA fixed"]["labels"],
    "M3 ExpA", "M2 V1", "IN test"
)

# ── M3 ExpA vs M3 Paper Style ─────────────────────────────────────
print("\n  M3 ExpA fixed vs M3 Paper Style:")
mcnemar_test(
    sf_results["M3 ExpA fixed"]["preds"],
    sf_results["M3 Paper Style"]["preds"],
    sf_results["M3 ExpA fixed"]["labels"],
    "M3 ExpA", "Paper Style", "SF test"
)
mcnemar_test(
    in_results["M3 ExpA fixed"]["preds"],
    in_results["M3 Paper Style"]["preds"],
    in_results["M3 ExpA fixed"]["labels"],
    "M3 ExpA", "Paper Style", "IN test"
)

# ── M3 ExpA vs M3 ExpC (LLRD) ────────────────────────────────────
print("\n  M3 ExpA fixed vs M3 ExpC (LLRD):")
mcnemar_test(
    sf_results["M3 ExpA fixed"]["preds"],
    sf_results["M3 ExpC (LLRD)"]["preds"],
    sf_results["M3 ExpA fixed"]["labels"],
    "M3 ExpA", "M3 ExpC", "SF test"
)
mcnemar_test(
    in_results["M3 ExpA fixed"]["preds"],
    in_results["M3 ExpC (LLRD)"]["preds"],
    in_results["M3 ExpA fixed"]["labels"],
    "M3 ExpA", "M3 ExpC", "IN test"
)
print("="*65)

  CORRECTED SIGNIFICANCE TESTS

  M3 ExpA fixed vs Vanilla FinBERT:
  SF test | M3 ExpA vs Vanilla
    b=125  c=26  p=0.0000  ✓ significant
  IN test | M3 ExpA vs Vanilla
    b=49  c=18  p=0.0002  ✓ significant

  M3 ExpA fixed vs M1 Exp2 (best M1):
  SF test | M3 ExpA vs M1 Exp2
    b=26  c=18  p=0.2913  ✗ not significant
  IN test | M3 ExpA vs M1 Exp2
    b=30  c=15  p=0.0369  ✓ significant

  M3 ExpA fixed vs M2 V1 (best M2):
  SF test | M3 ExpA vs M2 V1
    b=112  c=25  p=0.0000  ✓ significant
  IN test | M3 ExpA vs M2 V1
    b=15  c=17  p=0.8597  ✗ not significant

  M3 ExpA fixed vs M3 Paper Style:
  SF test | M3 ExpA vs Paper Style
    b=20  c=15  p=0.4990  ✗ not significant
  IN test | M3 ExpA vs Paper Style
    b=18  c=5  p=0.0123  ✓ significant

  M3 ExpA fixed vs M3 ExpC (LLRD):
  SF test | M3 ExpA vs M3 ExpC
    b=15  c=12  p=0.7003  ✗ not significant
  IN test | M3 ExpA vs M3 ExpC
    b=6  c=7  p=1.0000  ✗ not significant


In [20]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

def train_model_llrd(model_name, train_dataset, val_dataset,
                     output_dir, epochs=10,
                     base_lr=2e-5, decay=0.9,
                     dropout=0.2, weight_decay=0.1):

    print(f"\n{'█'*55}")
    print(f"  Training  : {model_name}")
    print(f"  Base LR   : {base_lr}  Decay: {decay}")
    print(f"  Dropout   : {dropout}")
    print(f"{'█'*55}\n")

    config = AutoConfig.from_pretrained(
        "ProsusAI/finbert",
        num_labels                   = 3,
        id2label                     = ID2LABEL,
        label2id                     = LABEL2ID,
        hidden_dropout_prob          = dropout,
        attention_probs_dropout_prob = dropout,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        "ProsusAI/finbert",
        config                  = config,
        ignore_mismatched_sizes = True
    ).to(device)

    optimizer_params = []

    # Classifier head — full learning rate
    optimizer_params.append({
        "params"      : [p for n, p in model.named_parameters()
                         if "classifier" in n],
        "lr"          : base_lr,
        "weight_decay": weight_decay
    })

    # BERT encoder layers — decay from top (layer 11) to bottom (layer 0)
    for layer_num in range(11, -1, -1):
        layer_lr = base_lr * (decay ** (11 - layer_num))
        optimizer_params.append({
            "params": [p for n, p in model.named_parameters()
                       if f"encoder.layer.{layer_num}." in n],
            "lr"    : layer_lr,
            "weight_decay": weight_decay
        })
        print(f"  Layer {layer_num:>2} lr: {layer_lr:.2e}")

    # Embeddings — smallest lr
    optimizer_params.append({
        "params"      : [p for n, p in model.named_parameters()
                         if "embeddings" in n],
        "lr"          : base_lr * (decay ** 12),
        "weight_decay": weight_decay
    })

    optimizer = AdamW(optimizer_params)

    args = TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = epochs,
        per_device_train_batch_size = 32,
        per_device_eval_batch_size  = 64,
        weight_decay                = 0,       # handled in optimizer
        warmup_steps                = 200,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        logging_steps               = 50,
        report_to                   = "none",
        seed                        = 42,
        fp16                        = True,
    )

    trainer = Trainer(
        model           = model,
        args            = args,
        train_dataset   = train_dataset,
        eval_dataset    = val_dataset,
        compute_metrics = compute_metrics,
        optimizers      = (optimizer, None),
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)]
    )

    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"\n  ✓ Saved → {output_dir}")
    return trainer


# ── Experiment A — M1 ────────────────────────────────────────────────────────
# Higher base LR, less aggressive decay, lighter regularisation.
# Rationale: M1 dataset is assumed cleaner/smaller — less regularisation needed.
trainer_m1_a = train_model_llrd(
    model_name    = "M1 — Exp A (LLRD)",
    train_dataset = ds_m1_train,
    val_dataset   = ds_m1_val,
    output_dir    = f"{MDIR}model1_expA",
    base_lr       = 3e-5,   # slightly higher — faster convergence on cleaner data
    decay         = 0.95,   # gentler decay — bottom layers learn more
    dropout       = 0.1,    # lighter dropout
    weight_decay  = 0.05,
    epochs        = 10
)

# ── Experiment B — M2 ────────────────────────────────────────────────────────
# Mid-point config between M1 and M3.
# More regularisation than M1, less aggressive decay than M3.
trainer_m2_b = train_model_llrd(
    model_name    = "M2 — Exp B (LLRD)",
    train_dataset = ds_m2_train,
    val_dataset   = ds_m2_val,
    output_dir    = f"{MDIR}model2_expB",
    base_lr       = 2e-5,   # same as M3 baseline
    decay         = 0.92,   # between M1 (0.95) and M3 (0.90)
    dropout       = 0.15,
    weight_decay  = 0.08,
    epochs        = 10
)



███████████████████████████████████████████████████████
  Training  : M1 — Exp A (LLRD)
  Base LR   : 3e-05  Decay: 0.95
  Dropout   : 0.1
███████████████████████████████████████████████████████



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Layer 11 lr: 3.00e-05
  Layer 10 lr: 2.85e-05
  Layer  9 lr: 2.71e-05
  Layer  8 lr: 2.57e-05
  Layer  7 lr: 2.44e-05
  Layer  6 lr: 2.32e-05
  Layer  5 lr: 2.21e-05
  Layer  4 lr: 2.10e-05
  Layer  3 lr: 1.99e-05
  Layer  2 lr: 1.89e-05
  Layer  1 lr: 1.80e-05
  Layer  0 lr: 1.71e-05


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.842115,0.593150,0.747600,0.746200
2,0.420835,0.402725,0.840500,0.840400
3,0.246662,0.446833,0.850700,0.851300
4,0.204719,0.446885,0.855600,0.855400
5,0.141529,0.608683,0.848900,0.849900


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  ✓ Saved → /content/drive/MyDrive/finbert_project/models/model2_expB
